# 01 – Bee Haven: Datenbereinigung (N1)

**Ziel:** Die Sensordaten aus `silver/processing` einheitlich und zuverlässig für die Analyse machen und nach `silver/cleaned` speichern.

**Regeln (Pflichtenheft PH-ITS-002, Schritte 2.4–2.6; Befund-Register Kap. 8.2):** R1 Zeitstempel → Datum/Uhrzeit · R2 Zeit → UTC · R3 fehlende Werte nur zählen · R4 flow-Duplikate behalten

**Grundsatz:** markieren statt löschen · `processing` wird nur gelesen, nie überschrieben

> **Synapse-Version (Azure Synapse Analytics)** – automatisch erzeugt aus `../01_bereinigung.ipynb` durch
> `Privat/03_notebooks_ITS-Data/generator/make_synapse_notebooks.py`. **Nicht von Hand ändern**, sondern das lokale
> Notebook ändern und den Generator erneut ausführen.
>
> Unterschiede zur lokalen Version (Vorbild: Kurs-Notebook 04 „weather pipeline synapse ready“):
> - Pfade als **ABFSS-URI (Azure Blob File System)** `abfss://<container>@<konto>.dfs.core.windows.net/...` statt `pathlib.Path`.
> - **Parameterzelle (Parameter Cell)** direkt darunter: eine Pipeline (Notebook-Aktivität) kann die Werte überschreiben.
> - **Kein Kontoschlüssel (Account Key) im Code:** Zugriff über die Identität des Synapse-Arbeitsbereichs (Workspace Identity)
>   bzw. einen verknüpften Dienst (Linked Service). Das Kurs-Notebook trägt den Schlüssel im Klartext ein – Sicherheitsrisiko.
> - Diagramme werden angezeigt; als PNG in den Lake nur bei `BILDER_SPEICHERN = True`.
> - Kosten: Ausführung nur nach Freigabe (Spark-Pool kostet je Minute).

In [ ]:
# --- Parameter (Parameters) – in Synapse als Parameterzelle markiert; Pipeline kann Werte überschreiben ---
STORAGE_ACCOUNT = "mabeehavenlakehouse"   # Speicherkonto (Storage Account, ADLS Gen2) des Lakehouse
CONTAINER = "lakehouse"                   # Container mit bronze/, silver/, gold/
STADT = "schwartau"                       # Standort = Unterordner in bronze/silver/gold und Teil der Dateinamen
LINKED_SERVICE = ""                       # leer = Identität des Arbeitsbereichs; sonst Name des verknüpften Dienstes (Linked Service)
BILDER_SPEICHERN = False                  # Diagramme zusätzlich als PNG nach gold/<stadt>/bilder/ schreiben?
LOKALER_TEST = ""                            # leer = Azure; lokaler Ordner (Spiegel des Containers) = Test ohne Azure

In [ ]:
# --- Hilfsfunktionen für Dateizugriffe: Azure Data Lake (abfss) oder lokaler Testordner ---
import os                                 # lokale Ordner (nur Testmodus)
import fnmatch                            # Dateinamen mit Platzhalter * vergleichen
import tempfile                           # Zwischenablage für PNG-Dateien

AZURE = not LOKALER_TEST                                                   # True = in Synapse gegen den Data Lake arbeiten
WURZEL = (f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"   # ABFSS-Wurzel des Containers (wie Kurs-Notebook 04)
          if AZURE else LOKALER_TEST.rstrip("/") + "/")                    # oder lokaler Testordner
SO = ({"linked_service": LINKED_SERVICE} if LINKED_SERVICE else {}) if AZURE else None   # storage_options für pandas
# Azure: SO nie None – sonst versucht pandas bei Parquet zuerst pyarrow direkt mit abfss:// und bricht ab
# ("Arrow compiled without Azure Blob File System support"); mit {} nimmt pandas den Synapse-Zugang über fsspec.
if AZURE:
    from notebookutils import mssparkutils                                 # Synapse-Werkzeuge: Dateien auflisten, Ordner anlegen, kopieren

def pfad(*teile):
    """Pfad unterhalb der Wurzel zusammensetzen, z. B. pfad("silver", "cleaned") -> abfss://.../silver/cleaned"""
    return WURZEL + "/".join(str(t).strip("/") for t in teile)

def dateien(ordner, muster="*"):
    """Sortierte Liste (voller Pfad, Größe in Byte) aller Dateien in EINEM Ordner, deren Name zum Muster passt."""
    if AZURE:
        try:
            eintraege = [(f.path, f.size) for f in mssparkutils.fs.ls(ordner) if f.isFile and fnmatch.fnmatch(f.name, muster)]
        except Exception:                                                  # Ordner existiert (noch) nicht
            eintraege = []
    else:
        eintraege = [(os.path.join(ordner, n), os.path.getsize(os.path.join(ordner, n)))
                     for n in (os.listdir(ordner) if os.path.isdir(ordner) else []) if fnmatch.fnmatch(n, muster)]
    return sorted(eintraege)                                               # sortiert nach Name -> letzte = neueste Version (Zeitstempel)

def neueste(ordner, muster):
    """Pfad der jüngsten Datei (Name enthält UTC-Zeitstempel) – Fehler, wenn keine vorhanden."""
    treffer = dateien(ordner, muster)
    if not treffer:
        raise FileNotFoundError(f"keine Datei {muster} in {ordner}")
    return treffer[-1][0]

def ordner_anlegen(ordner):
    """Zielordner anlegen, falls nicht vorhanden (in ADLS Gen2 unschädlich, wenn schon da)."""
    mssparkutils.fs.mkdirs(ordner) if AZURE else os.makedirs(ordner, exist_ok=True)

def archivieren(datei, neuer_name=None):
    """Datei in das _archiv ihres eigenen Ordners verschieben (Regel: jeder Ordner hat ein _archiv) – nichts wird gelöscht."""
    ordner, name = datei.rsplit("/", 1)
    ziel = f"{ordner}/_archiv/{neuer_name or name}"
    ordner_anlegen(f"{ordner}/_archiv")
    mssparkutils.fs.mv(datei, ziel) if AZURE else os.replace(datei, ziel)

def groesse_mb(datei):
    """Dateigröße in Megabyte (zur Kontrolle nach dem Speichern)."""
    ordner, name = datei.rsplit("/", 1)
    return next((g for p, g in dateien(ordner, name)), 0) / 1_000_000

def bild_speichern(fig, name, **optionen):
    """Diagramm optional als PNG in den Lake schreiben (Parameter BILDER_SPEICHERN)."""
    if not BILDER_SPEICHERN:
        return
    ziel_ordner = pfad("gold", STADT, "bilder")
    ordner_anlegen(ziel_ordner)
    lokal = os.path.join(tempfile.gettempdir(), name)                      # erst lokal auf dem Knoten speichern ...
    fig.savefig(lokal, **optionen)
    if AZURE:
        mssparkutils.fs.cp("file:" + lokal, ziel_ordner + "/" + name)       # ... dann in den Data Lake kopieren
    else:
        os.replace(lokal, os.path.join(ziel_ordner, name))

print("Wurzel:", WURZEL, "| Azure:", AZURE)

## 0. Bibliotheken und Pfade

In [ ]:
# --- Bibliotheken laden ---
import pandas as pd                      # pandas: Tabellen (DataFrames) einlesen, bereinigen, speichern
from datetime import datetime, timezone  # für den UTC-Zeitstempel im Dateinamen der Ausgabe

# --- Pfade festlegen (Struktur im Container lakehouse) ---
PROCESSING = pfad("silver", "processing", STADT)          # Eingang: unbereinigte Arbeitskopie (nur LESEN!)
CLEANED = pfad("silver", "cleaned", STADT)                # Ausgang: bereinigte Dateien mit Zeitstempel
ZWISCHENSTAND = pfad("silver", "zwischenstand", STADT)    # Zwischenergebnis N1 -> N2/N3 (lokal: 02_daten_ITS-Data/zwischenstand)

# --- Standortdaten aus der Konfiguration (Koordinaten, Zeitzone) ---
standorte = pd.read_csv(pfad("konfiguration", "standorte.csv"), index_col="stadt", storage_options=SO)
if STADT not in standorte.index:                          # unbekannte Stadt -> sofort abbrechen statt mit falschen Daten weiterrechnen
    raise ValueError(f"Standort '{STADT}' fehlt in konfiguration/standorte.csv – Zeile mit Koordinaten und Zeitzone ergänzen")
ORT = standorte.loc[STADT]                                # eine Zeile: name, breite, laenge, zeitzone, quelle
ZEITZONE = ORT["zeitzone"]                                # Ortszeit der Rohdaten (Deutschland: Europe/Berlin)

# --- Sensor-Regeln je Standort (Einheit der Rohdaten, gültiger Bereich) und bekannte Ausfallzeiträume ---
sensoren_cfg = pd.read_csv(pfad("konfiguration", "sensoren.csv"), storage_options=SO)          # eine Zeile je Stadt und Sensor
REGELN = sensoren_cfg[sensoren_cfg["stadt"] == STADT].set_index("sensor")              # Regeln dieser Stadt: einheit_roh, min, max
ausschluesse = pd.read_csv(pfad("konfiguration", "ausschluesse.csv"), storage_options=SO)      # bekannte Ausfallzeiträume
AUSSCHLUESSE = ausschluesse[ausschluesse["stadt"] == STADT].copy()                     # nur Zeiträume dieser Stadt
AUSSCHLUESSE[["von_utc", "bis_utc"]] = AUSSCHLUESSE[["von_utc", "bis_utc"]].apply(pd.to_datetime, utc=True)  # Text -> UTC-Zeit

# --- Kontrolle: Welche Dateien liegen im Eingang? ---
for s in ["flow", "humidity", "temperature", "weight"]:   # je Sensor-Unterordner ...
    for datei, groesse in dateien(f"{PROCESSING}/{s}", "*.csv"):   # ... alle CSV auflisten
        print(datei.replace(WURZEL, ""), f"{groesse / 1_000_000:.1f} MB")  # Pfad (ab Container) und Größe ausgeben

**Ergebnis:** 

Alle 4 Dateien liegen im Eingang `processing/schwartau/<sensor>/`: flow 56,9 MB · humidity 0,0 MB · temperature 6,5 MB · weight 0,1 MB.

## 1. Daten einlesen (Original)

In [ ]:
# --- Alle 4 CSV-Dateien einlesen ---
sensoren = ["flow", "humidity", "temperature", "weight"]  # Liste der Sensoren (= Ordnernamen in processing)
df = {}                                                   # leeres Dictionary: Sensorname -> Tabelle (DataFrame)
for s in sensoren:                                        # für jeden Sensor nacheinander ...
    datei = f"{PROCESSING}/{s}/{s}_{STADT}.csv"           # ... den ABFSS-Pfad zusammensetzen
    df[s] = pd.read_csv(datei, storage_options=SO)        # ... die CSV als Tabelle einlesen (nur lesen!)
    print(f"{s:12} {len(df[s]):>10,} Zeilen   Spalten: {list(df[s].columns)}")  # Kontrolle ausgeben

**Ergebnis:** 

flow 2.513.836 · humidity 1.761 · temperature 253.430 · weight 1.761 Zeilen; jeweils 2 Spalten (`timestamp` + Messwert).

## 2. Arbeitskopie erstellen

In [ ]:
# --- Arbeitskopien anlegen: Die Originale in df bleiben zum Vergleich unverändert ---
cl = {s: tabelle.copy() for s, tabelle in df.items()}  # cl = "cleaned" (Namenskonvention aus der Kursvorlage)
print("Arbeitskopien:", list(cl.keys()))               # Kontrolle: Für welche Sensoren gibt es eine Kopie?

# --- Einheit vereinheitlichen: Gewicht intern immer in Gramm (Schwartau liefert g, Würzburg kg; Quelle: konfiguration/sensoren.csv) ---
FAKTOR_ZU_GRAMM = {"g": 1, "kg": 1000}                                    # Umrechnungsfaktor je Einheit der Rohdaten
einheit = REGELN.loc["weight", "einheit_roh"]                              # Einheit der Rohdaten dieser Stadt
cl["weight"]["weight"] = cl["weight"]["weight"] * FAKTOR_ZU_GRAMM[einheit]  # z. B. 52,7 kg -> 52.700 g
print(f"weight: Rohdaten in {einheit} -> intern in g (Faktor {FAKTOR_ZU_GRAMM[einheit]})")  # Kontrolle

**Ergebnis:** 

Arbeitskopien für alle 4 Sensoren angelegt (`cl`); die Originale (`df`) bleiben zum Vergleich unverändert.

## 3. Exploration der Daten

In [ ]:
# --- Überblick je Sensor: Beispielzeilen, Datentypen, fehlende Werte, Wertebereich ---
for s in sensoren:                                         # für jeden Sensor nacheinander ...
    print(f"===== {s} =====")                              # ... Überschrift ausgeben
    print(cl[s].head(3))                                   # ... die ersten 3 Zeilen anzeigen
    print("Datentypen:", dict(cl[s].dtypes.astype(str)))   # ... Datentyp je Spalte (timestamp ist noch Text!)
    print("Fehlende Werte:", dict(cl[s].isna().sum()))     # ... Anzahl leerer Zellen je Spalte zählen
    print(cl[s][s].describe().round(2).to_dict())          # ... Min, Max, Mittelwert usw. der Messwert-Spalte
    print()                                                # ... Leerzeile für bessere Lesbarkeit

**Ergebnis:** 

Befunde aus dem Pflichtenheft bestätigt: 

`timestamp` ist Text (R1) · fehlende Werte 0 / 12 / 2.032 / 12 (R3) · Minimum 

flow −631 (2 Messreihen, R8) · humidity −100 (R6) · temperature −7,26 (R7) · weight −172,44 (R5).

## 4. Datentypen und Zeitzone (R1, R2)
**R1:** `timestamp` ist Text → in Datum/Uhrzeit umwandeln, sonst kann man nicht sortieren, gruppieren oder rechnen.

**R2:** Die Zeiten sind deutsche Ortszeit mit Sommerzeit. Im Oktober gibt es 02:00–03:00 Uhr doppelt → Umrechnung in **UTC** (ohne Sommerzeit, eindeutig).

Die Originalspalte `timestamp` bleibt erhalten (markieren statt löschen); neu ist die Spalte `zeit_utc`.

In [ ]:
# --- Funktion: deutsche Ortszeit (Text) -> eindeutige UTC-Zeit ---
def zu_utc(zeit_text):
    """Wandelt Zeitstempel-Text in deutscher Ortszeit (mit Sommerzeit) in UTC um."""
    zeit = pd.to_datetime(zeit_text, format="%Y-%m-%d %H:%M:%S")  # R1: Text -> Datum/Uhrzeit (Format fest vorgegeben)
    neustart = (zeit.diff() < pd.Timedelta(hours=-2)).cumsum()  # Zeit springt weit zurück = neue Messreihe (flow hat 2)
    wiederholung = zeit.groupby([neustart, zeit]).cumcount()    # 0 = Uhrzeit kommt zum 1. Mal vor, 1 = zum 2. Mal
    sommerzeit = (wiederholung == 0).to_numpy()                 # 1. Vorkommen = noch Sommerzeit, 2. = schon Winterzeit
    zeit = zeit.dt.tz_localize(ZEITZONE,                        # R2: Zeitzone des Standorts zuordnen (Standort-Tabelle) ...
                               ambiguous=sommerzeit,            # ... doppelte Oktober-Stunde nach der Regel oben auflösen
                               nonexistent="raise")             # ... Fehler melden, falls eine Zeit in der März-Lücke liegt
    return zeit.dt.tz_convert("UTC")                            # ... und in UTC umrechnen (keine Sommerzeit)

In [ ]:
# --- R1 + R2 auf alle Sensoren anwenden ---
for s in sensoren:                                           # für jeden Sensor ...
    cl[s]["zeit_utc"] = zu_utc(cl[s]["timestamp"])           # ... neue Spalte zeit_utc (Original timestamp bleibt!)
    print(f"{s:12} erste Zeit: {cl[s]['zeit_utc'].iloc[0]}   Typ: {cl[s]['zeit_utc'].dtype}")  # Kontrolle

# --- Beweis: humidity misst immer zur gleichen UTC-Uhrzeit (lokal: Sommer 02/14 Uhr, Winter 01/13 Uhr) ---
print("humidity UTC-Uhrzeiten:", sorted(cl["humidity"]["zeit_utc"].dt.strftime("%H:%M").unique()))

**Ergebnis:** 

Alle 4 Sensoren haben die neue Spalte `zeit_utc` vom Typ `datetime64[us, UTC]`. 

humidity misst nach UTC immer um 00:00 und 12:00 Uhr (lokal Sommer 02/14 Uhr, Winter 01/13 Uhr) 

→ Beweis, dass die Rohdaten deutsche Ortszeit mit Sommerzeit sind und die Umrechnung stimmt (Pflichtenheft E2).

## 5. Duplikate (R4)

Jetzt, **nach** der UTC-Umrechnung, lassen sich Duplikate richtig beurteilen (Befund E1).

Prüfung 1: Gibt es noch doppelte Zeiten innerhalb einer Messreihe? → erwartet: 0

Prüfung 2: Wie viele komplett gleiche Zeilen haben die Rohdaten? → flow ca. 697.000 = echte Minuten mit dem Wert 0 → **behalten**

In [ ]:
# --- Duplikate prüfen (nach der UTC-Umrechnung) ---
for s in sensoren:                                                        # für jeden Sensor ...
    reihe = (cl[s]["zeit_utc"].diff() < pd.Timedelta(hours=-2)).cumsum()  # ... Messreihe bestimmen (flow: 0 und 1)
    doppelte_zeit = cl[s].assign(reihe=reihe).duplicated(subset=["reihe", "zeit_utc"]).sum()  # ... gleiche Zeit in derselben Reihe zählen
    gleiche_rohzeilen = df[s].duplicated().sum()                          # ... komplett gleiche Zeilen in den ROHdaten zählen
    print(f"{s:12} doppelte UTC-Zeiten: {doppelte_zeit:>3}   gleiche Rohzeilen: {gleiche_rohzeilen:>9,}")  # ... Ergebnis ausgeben


**Ergebnis:** 

Doppelte UTC-Zeiten: überall 0 → die 24 „Duplikate“ bei temperature entstanden nur durch die Zeitumstellung (E1, E4). 

Gleiche Rohzeilen: nur flow mit 697.348 = echte Minuten mit dem Wert 0 → werden behalten (R4).

## 5a. Zeitabstände und versteckte Lücken
**Warum:** Eine Messreihe kann vollständig aussehen und trotzdem Lücken haben – dann fehlen Zeilen, statt dass NaN drinsteht. Das sieht man nur an den Zeitabständen.
**Eigene Erwartung:** flow misst jede Minute; wir erwarten einige Lücken, darunter längere Ausfälle. Die anderen Sensoren sollten lückenlos sein.

In [ ]:
# --- Zeitabstände je Messreihe prüfen (nach UTC) ---
luecken_protokoll = []                                                  # Sammlung der Ergebnisse
for s in sensoren:                                                      # für jeden Sensor ...
    zeit = cl[s]["zeit_utc"]                                            # ... UTC-Zeitspalte
    reihe = (zeit.diff() < pd.Timedelta(hours=-2)).cumsum()             # ... Messreihe (flow: 0 = Ausflug, 1 = Einflug)
    abstand = zeit.groupby(reihe).diff().dropna()                       # ... Abstand zur vorherigen Zeile derselben Reihe
    normal = abstand.mode()[0]                                          # ... häufigster Abstand = normales Messintervall
    zu_gross = abstand[abstand > normal]                                # ... alle Abstände größer als normal = Lücken
    luecken_protokoll.append({"sensor": s, "normal": str(normal),       # ... Ergebnis festhalten:
                              "luecken": len(zu_gross),                 #     Anzahl Lücken
                              "davon_ueber_1_tag": int((zu_gross > pd.Timedelta(days=1)).sum()),
                              "groesste": str(zu_gross.max()) if len(zu_gross) else "-"})
pd.DataFrame(luecken_protokoll)                                         # Übersicht als Tabelle

**Ergebnis:**

flow (normal 1 Minute): **120 versteckte Lücken**, davon **4 länger als 1 Tag**, größte **5 Tage 09:32** → Kapitel 6 zeigt bei flow 0 fehlende Werte, die Lücken sieht man nur hier.

humidity/weight (12 Stunden) und temperature (5 Minuten): lückenlos.

→ Umgang mit den Lücken (kleine auffüllen oder nur markieren) entscheiden wir in N2 (Pflichtenheft P6).

## 6. Fehlende Werte (R3)

**Regel:** nur zählen und dokumentieren, **nichts auffüllen**.

**Warum?** Erfundene Werte (z. B. ein Mittelwert) täuschen Messungen vor, die es nie gab. Wie mit Lücken umgegangen wird, entscheidet erst Gold (N4) bzw. die Analyse (N5) – Entscheidung P6 = c: in Silver nichts auffüllen (Pflichtenheft Schritt 3.1).

In [ ]:
# --- Fehlende Werte zählen (R3: nur zählen, nichts auffüllen) ---
protokoll = []                                                # leere Liste für das Bereinigungsprotokoll
for s in sensoren:                                            # für jeden Sensor ...
    fehlend = int(cl[s][s].isna().sum())                      # ... leere Messwerte zählen
    anteil = fehlend / len(cl[s]) * 100                       # ... Anteil in Prozent berechnen
    protokoll.append({"sensor": s,                            # ... eine Zeile ins Protokoll schreiben:
                      "zeilen": len(cl[s]),                   #     Anzahl Zeilen
                      "fehlend": fehlend,                     #     Anzahl fehlender Werte
                      "anteil_%": round(anteil, 2)})          #     Anteil in Prozent (2 Nachkommastellen)
pd.DataFrame(protokoll)                                       # Protokoll als Tabelle anzeigen

**Ergebnis:** 

Fehlende Messwerte: flow 0 · humidity 12 (0,68 %) · temperature 2.032 (0,80 %) · weight 12 (0,68 %). 

Nicht aufgefüllt, nur dokumentiert (R3).

## 7. Speichern (Abschluss N1)

**Ziel:** Das Ergebnis von N1 als Zwischenstand für N2 (Datenqualität) sichern.

**Wohin?** `02_daten_ITS-Data/zwischenstand/` – nicht nach `processing` (wird nie überschrieben) und noch nicht nach `cleaned` (erst nach N2).

**Warum Parquet statt CSV?** Parquet behält die Datentypen (z. B. `zeit_utc` als UTC-Zeit), CSV macht alles wieder zu Text. Außerdem ca. halb so groß und schneller.

**Kontrollwert (Ablage laut Plan):** 4 Dateien `<sensor>_<stadt>_n1.parquet` + `n1_protokoll.csv`

In [ ]:
# --- N1-Ergebnis speichern (Zwischenstand für N2) ---
ordner_anlegen(ZWISCHENSTAND)                                # Ordner anlegen, falls er noch nicht existiert
for s in sensoren:                                           # für jeden Sensor ...
    ziel = f"{ZWISCHENSTAND}/{s}_{STADT}_n1.parquet"         # ... Dateiname festlegen
    cl[s].to_parquet(ziel, index=False, storage_options=SO)  # ... als Parquet speichern (behält Datentypen inkl. UTC)
    print(f"{s:12} gespeichert: {ziel.replace(WURZEL, '')}  ({groesse_mb(ziel):.1f} MB)")  # ... Kontrolle ausgeben
pd.DataFrame(protokoll).to_csv(f"{ZWISCHENSTAND}/n1_protokoll.csv", index=False, storage_options=SO)  # Protokoll sichern
print("Protokoll gespeichert: n1_protokoll.csv")                                        # Kontrolle

**Ergebnis:**

4 Parquet-Dateien in `02_daten_ITS-Data/zwischenstand/`: flow 28,2 MB · humidity 0,0 MB · temperature 3,4 MB · weight 0,0 MB (Parquet ca. halb so groß wie CSV).

Protokoll `n1_protokoll.csv` gespeichert. **N1 abgeschlossen** → weiter mit N2.